In [2]:
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ================= CONFIGURATION =================
MODEL_NAME = "roberta-base"
MAX_LEN = 128
BATCH_SIZE = 32
VAL_BATCH_SIZE = 16
TEST_BATCH_SIZE = 4
EPOCHS = 5
LR = 2e-5
PATIENCE = 5  # Early stopping patience
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_COLS = ['anger', 'fear', 'joy', 'sadness', 'surprise']
PATHS = {
    'train': '/kaggle/input/competitions/2025-sep-dl-gen-ai-project/train.csv',
    'test': '/kaggle/input/competitions/2025-sep-dl-gen-ai-project/test.csv',
    'sample': '/kaggle/input/competitions/2025-sep-dl-gen-ai-project/sample_submission.csv',
    'best_model': 'best_roberta_emotion.pth'
}

# ================= UTILS =================
def clean_text(text):
    return str(text).strip().lower()

# ================= DATASET =================
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, is_test=False):
        self.texts = df["text"].values
        self.labels = df[LABEL_COLS].values if not is_test and set(LABEL_COLS).issubset(df.columns) else None
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# ================= MODEL =================
class EmotionClassifier(nn.Module):
    def __init__(self, model_name, num_labels=5):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0]
        return self.classifier(self.dropout(pooled))

# ================= TRAINING HELPERS =================
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, device):
    model.eval()
    preds_list, true_list = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].cpu().numpy()

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds_list.append(probs)
            true_list.append(labels)
    return np.vstack(true_list), np.vstack(preds_list)

def optimize_thresholds(y_true, y_probs, label_cols):
    """Finds optimal per-class threshold to maximize Macro F1 on validation set."""
    best_thresholds = np.ones(len(label_cols)) * 0.5
    for i in range(len(label_cols)):
        best_f1, best_thresh = -1, 0.5
        for thresh in np.arange(0.1, 0.9, 0.01):
            preds = (y_probs[:, i] > thresh).astype(int)
            f1 = f1_score(y_true[:, i], preds)
            if f1 > best_f1:
                best_f1, best_thresh = f1, thresh
        best_thresholds[i] = best_thresh
    return best_thresholds

# ================= PIPELINES =================
def train_pipeline():
    print(">> Loading data...")
    train_df = pd.read_csv(PATHS['train'])
    train_df['text'] = train_df['text'].apply(clean_text)

    train_split, val_split = train_test_split(train_df, test_size=0.1, random_state=42, shuffle=True)
    print(f"Train: {train_split.shape}, Val: {val_split.shape}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_loader = DataLoader(EmotionDataset(train_split, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader   = DataLoader(EmotionDataset(val_split, tokenizer, MAX_LEN), batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0)

    model = EmotionClassifier(MODEL_NAME).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * 0.1)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    best_f1, patience_counter = 0, 0
    best_thresholds = np.ones(len(LABEL_COLS)) * 0.5

    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        avg_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, DEVICE)
        y_true, y_probs = eval_epoch(model, val_loader, DEVICE)
        
        # Track with 0.5 threshold for early stopping
        val_f1 = f1_score(y_true, (y_probs > 0.5).astype(int), average="macro")
        print(f"Loss: {avg_loss:.4f} | Val F1 (t=0.5): {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), PATHS['best_model'])
            print(f">> New best F1 saved: {best_f1:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(">> Early stopping triggered.")
                break

    # Optimize thresholds on validation set
    print("\n>> Optimizing decision thresholds...")
    best_thresholds = optimize_thresholds(y_true, y_probs, LABEL_COLS)
    print(f"Optimal Thresholds: {dict(zip(LABEL_COLS, best_thresholds.round(2)))}")
    np.save("optimal_thresholds.npy", best_thresholds)
    print(">> Training complete.")

def inference_pipeline():
    print(">> Loading data & model...")
    test_df = pd.read_csv(PATHS['test'])
    test_df['text'] = test_df['text'].apply(clean_text)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    test_loader = DataLoader(EmotionDataset(test_df, tokenizer, MAX_LEN, is_test=True), batch_size=TEST_BATCH_SIZE, shuffle=False, num_workers=0)

    model = EmotionClassifier(MODEL_NAME).to(DEVICE)
    if os.path.exists(PATHS['best_model']):
        model.load_state_dict(torch.load(PATHS['best_model'], map_location=DEVICE, weights_only=True))
        print(">> Loaded best model weights.")
    else:
        raise FileNotFoundError("No trained model found. Run training first.")

    model.eval()
    probs_all = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Inference"):
            logits = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
            probs_all.append(torch.sigmoid(logits).cpu().numpy())
    probs_all = np.vstack(probs_all)

    thresholds = np.load("optimal_thresholds.npy") if os.path.exists("optimal_thresholds.npy") else np.ones(len(LABEL_COLS)) * 0.5
    preds_all = (probs_all > thresholds).astype(int)

    submission = pd.read_csv(PATHS['sample'])
    submission[LABEL_COLS] = preds_all
    submission.to_csv("submission.csv", index=False)
    print(">> Submission saved to submission.csv")
    print(submission.head())

# ================= MAIN =================
IS_TRAINING_MODE = True 

if __name__ == "__main__":
    start_time = time.time()
    print("=" * 60)
    if IS_TRAINING_MODE:
        print(">> MODE: TRAINING")
        train_pipeline()
    else:
        print(">> MODE: INFERENCE (Submission Ready)")
        inference_pipeline()
    
    total_time = time.time() - start_time
    print("=" * 60)
    print(f">> Total Execution Time: {total_time:.2f}s")
    print("=" * 60)

>> MODE: TRAINING
>> Loading data...
Train: (6144, 8), Val: (683, 8)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Epoch 1/5


Evaluating: 100%|██████████| 43/43 [00:05<00:00,  8.25it/s]


...

Loss: 0.2305 | Val F1 (t=0.5): 0.7874
>> New best F1 saved: 0.7874

Epoch 4/5


Evaluating: 100%|██████████| 43/43 [00:05<00:00,  8.31it/s]


Loss: 0.1765 | Val F1 (t=0.5): 0.8262
>> New best F1 saved: 0.8262

Epoch 5/5


Evaluating: 100%|██████████| 43/43 [00:05<00:00,  8.29it/s]


Loss: 0.1443 | Val F1 (t=0.5): 0.8472
>> New best F1 saved: 0.8472

>> Optimizing decision thresholds...
Optimal Thresholds: {'anger': np.float64(0.49), 'fear': np.float64(0.44), 'joy': np.float64(0.23), 'sadness': np.float64(0.24), 'surprise': np.float64(0.35)}
>> Training complete.
>> Total Execution Time: 708.03s
